In [48]:
import csv
from xmlrpc.client import boolean

import pandas as pd
import torch
df_train = pd.read_csv("transformedtest2.csv")
df_test = pd.read_csv("transformedtest2.csv")
n_features = len(df_train.columns) -2


In [49]:
from sklearn.model_selection import train_test_split
X = df_train.drop(["Y","ID"],axis=1)
y = df_train["Y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [50]:
from xgboost import XGBClassifier  # or XGBRegressor if it's regression
from sklearn.metrics import f1_score
# Train XGBoost classifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')  # disable warning
xgb.fit(X_train, y_train)
# Predict and evaluate
y_xgb = xgb.predict(X_test)
f1_xgb = f1_score(y_test, y_xgb, average='weighted')  # or 'macro' depending on your case
print(f"F1 Score: {f1_xgb:.4f}")

/media/josh/LARGEHDD/Projects/InsightFactoryHackathon2025/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [05:27:55] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


F1 Score: 0.7552


In [51]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier,ExtraTreesClassifier

rcf = RandomForestClassifier(random_state=0)
rcf.fit(X_train, y_train)
y_rcf= rcf.predict(X_test)
f1_rcf = f1_score(y_test, y_rcf, average='weighted')
print(f'The F1 Score is: {f1_rcf:.4f}')


The F1 Score is: 0.7468


In [52]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import f1_score

# Initialize the AdaBoost classifier
# You can specify n_estimators (number of weak learners) and learning_rate
ada = AdaBoostClassifier(random_state=0)

# Train the model
ada.fit(X_train, y_train)

# Make predictions
y_ada = ada.predict(X_test)

# Calculate F1 score
f1_ada = f1_score(y_test, y_ada, average='weighted')
print(f'The F1 Score is: {f1_ada:.4f}')

The F1 Score is: 0.7379


In [53]:


def combine_results(*args):
    n = len(args)
    total = np.sum(args,axis=0)
    plurality =n - n//2
    consensus = (total>=plurality).astype(int)
    return consensus

y_pred_stacked = combine_results(y_xgb,y_rcf,y_ada)
f1_score(y_test, y_pred_stacked, average='weighted')



0.7446997724103994

Below this is a graveyard, a failed attempt to make a neural network work with this stacked model system

In [43]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Define the model
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 8),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Create model instance
NN = BinaryClassifier(input_dim=n_features)

# Define loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy
optimizer = optim.Adam(NN.parameters())

import matplotlib.pyplot as plt
def train_NN(X_train, y_train, X_test=None, y_test=None, epochs=5, batch_size=10):
    # Convert training data to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train).view(-1, 1)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Convert validation data if provided
    if X_test is not None and y_test is not None:
        X_test_tensor = torch.FloatTensor(X_test)
        y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        NN.train()
        running_loss = 0.0

        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = NN(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Compute validation loss if validation data is provided
        if X_test is not None and y_test is not None:
            NN.eval()
            with torch.no_grad():
                outputs = NN(X_test_tensor)
                val_loss = criterion(outputs, y_test_tensor).item()
                val_losses.append(val_loss)
        else:
            val_losses.append(None)

        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}', end='')
        if val_losses[-1] is not None:
            print(f', Val Loss: {val_losses[-1]:.4f}')
        else:
            print()

    # Plot losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    if any(val is not None for val in val_losses):
        plt.plot([v if v is not None else float('nan') for v in val_losses], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss over Epochs')
    plt.legend()
    plt.grid(True)
    plt.show()



# Evaluation function
def transform_NN(X):
    NN.eval()
    X_test_tensor = torch.FloatTensor(X)

    with torch.no_grad():
        outputs = NN(X)
        predicted = (outputs > 0.5).int()
        return predicted

train_NN(np.asarray(X_train), np.asarray(y_train), np.asarray(X_test), np.asarray(y_test), epochs=20, batch_size=10)


Epoch 1/20, Train Loss: 5.6720, Val Loss: 0.4934
Epoch 2/20, Train Loss: 0.5454, Val Loss: 0.4935
Epoch 3/20, Train Loss: 0.4970, Val Loss: 0.4935
Epoch 4/20, Train Loss: 0.4954, Val Loss: 0.4934
Epoch 5/20, Train Loss: 0.5016, Val Loss: 0.4935
Epoch 6/20, Train Loss: 0.4959, Val Loss: 0.4934
Epoch 7/20, Train Loss: 0.4975, Val Loss: 0.4936
Epoch 8/20, Train Loss: 0.4954, Val Loss: 0.4934
Epoch 9/20, Train Loss: 0.4954, Val Loss: 0.4934
Epoch 10/20, Train Loss: 0.4954, Val Loss: 0.4935
Epoch 11/20, Train Loss: 0.4954, Val Loss: 0.4934
Epoch 12/20, Train Loss: 0.4954, Val Loss: 0.4934
Epoch 13/20, Train Loss: 0.4954, Val Loss: 0.4934


KeyboardInterrupt: 

In [33]:
from sklearn.metrics import f1_score
def evaluate(X_test, y_test):
    NN.eval()
    X_test_tensor = torch.FloatTensor(X_test)

    with torch.no_grad():
        outputs = NN(X_test_tensor)
        predicted = (outputs > 0.5).float().numpy().flatten()

    y_true = np.array(y_test).flatten()

    # Compute F1 score
    f1 = f1_score(y_true, predicted,average='weighted')

    print(f'F1 Score: {f1:.4f}')
evaluate(X_test, y_test)

F1 Score: 0.7267
